In [3]:
!pip install duckdb pandas pyarrow -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving team_4.parquet to team_4.parquet


In [3]:
import duckdb
import pandas as pd
import time

con = duckdb.connect("air_quality.duckdb")
print("✓ Connected to DuckDB")

✓ Connected to DuckDB


In [4]:
con.sql("DESCRIBE SELECT * FROM 'team_4.parquet'").show()

┌──────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │       column_type        │  null   │   key   │ default │  extra  │
│   varchar    │         varchar          │ varchar │ varchar │ varchar │ varchar │
├──────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ station_id   │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ state        │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ city         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ station_name │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ timestamp    │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ datetime     │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ at_c         │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NULL    │
│ rh_percent   │ DOUBLE                   │ YES     │ NULL    │ NULL    │ NU

In [5]:
con.sql("DROP TABLE IF EXISTS measurements")
con.sql("DROP TABLE IF EXISTS stations")
con.sql("DROP TABLE IF EXISTS pollutants")

con.sql("""
CREATE TABLE stations (
    station_id   VARCHAR PRIMARY KEY,
    city         VARCHAR,
    state        VARCHAR
)
""")

con.sql("""
CREATE TABLE pollutants (
    pollutant_id    INTEGER PRIMARY KEY,
    pollutant_name  VARCHAR UNIQUE
)
""")

con.sql("""
CREATE TABLE measurements (
    measurement_id  BIGINT,
    datetime        TIMESTAMP,
    station_id      VARCHAR,
    pollutant_id    INTEGER,
    value           DOUBLE,
    year            INTEGER,
    month           INTEGER,
    day             INTEGER,
    hour            INTEGER,
    at_c            DOUBLE,
    rh_percent      DOUBLE,
    bp_mmhg         DOUBLE,
    ws_m_s          DOUBLE,
    wd_deg          DOUBLE,
    rf_mm           DOUBLE,
    sr_w_mt2        DOUBLE,
    vws_m_s         DOUBLE
)
""")

print("✓ Schema created")
con.sql("SHOW TABLES").show()

✓ Schema created
┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ measurements │
│ pollutants   │
│ stations     │
└──────────────┘



In [6]:
con.sql("""
INSERT INTO stations
SELECT DISTINCT station_id, city, state
FROM 'team_4.parquet'
""")

con.sql("""
INSERT INTO pollutants
SELECT
    ROW_NUMBER() OVER (ORDER BY pollutant) AS pollutant_id,
    pollutant AS pollutant_name
FROM (SELECT DISTINCT pollutant FROM 'team_4.parquet')
""")

print("Stations:")
con.sql("SELECT * FROM stations").show()
print("Pollutants:")
con.sql("SELECT * FROM pollutants").show()

Stations:
┌────────────┬─────────┬─────────┐
│ station_id │  city   │  state  │
│  varchar   │ varchar │ varchar │
├────────────┼─────────┼─────────┤
│ site_104   │ Delhi   │ Delhi   │
│ site_108   │ Delhi   │ Delhi   │
│ site_118   │ Delhi   │ Delhi   │
│ site_1420  │ Delhi   │ Delhi   │
│ site_301   │ Delhi   │ Delhi   │
│ site_5393  │ Delhi   │ Delhi   │
│ site_1421  │ Delhi   │ Delhi   │
│ site_1560  │ Delhi   │ Delhi   │
│ site_5024  │ Delhi   │ Delhi   │
│ site_103   │ Delhi   │ Delhi   │
├────────────┴─────────┴─────────┤
│ 10 rows              3 columns │
└────────────────────────────────┘

Pollutants:
┌──────────────┬────────────────┐
│ pollutant_id │ pollutant_name │
│    int32     │    varchar     │
├──────────────┼────────────────┤
│            1 │ benzene        │
│            2 │ co             │
│            3 │ eth_benzene    │
│            4 │ mp_xylene      │
│            5 │ nh3            │
│            6 │ no             │
│            7 │ no2            │
│       

In [7]:
start = time.time()

con.sql("""
INSERT INTO measurements
SELECT
    ROW_NUMBER() OVER () AS measurement_id,
    m.datetime,
    m.station_id,
    p.pollutant_id,
    m.value,
    m.year, m.month, m.day, m.hour,
    m.at_c, m.rh_percent, m.bp_mmhg,
    m.ws_m_s, m.wd_deg, m.rf_mm,
    m.sr_w_mt2, m.vws_m_s
FROM 'team_4.parquet' AS m
JOIN pollutants AS p ON m.pollutant = p.pollutant_name
""")

load_time = time.time() - start
print(f"✓ Loaded measurements in {load_time:.2f} seconds")

count = con.sql("SELECT COUNT(*) AS n FROM measurements").fetchone()[0]
print(f"  Total rows: {count:,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✓ Loaded measurements in 11.70 seconds
  Total rows: 5,831,431


In [8]:
def timed_query(label, sql):
    start = time.time()
    result = con.sql(sql).df()
    elapsed = (time.time() - start) * 1000
    print(f"⏱  {label}: {elapsed:.1f} ms  ({len(result):,} rows)")
    return result

q1 = timed_query("Avg PM2.5 per month", """
    SELECT year, month, AVG(value) AS avg_pm25
    FROM measurements m
    JOIN pollutants p ON m.pollutant_id = p.pollutant_id
    WHERE p.pollutant_name = 'pm25'
    GROUP BY year, month
    ORDER BY year, month
""")

q2 = timed_query("Top 3 stations by avg PM10", """
    SELECT s.station_id, s.city, AVG(m.value) AS avg_pm10
    FROM measurements m
    JOIN pollutants p ON m.pollutant_id = p.pollutant_id
    JOIN stations s   ON m.station_id   = s.station_id
    WHERE p.pollutant_name = 'pm10'
    GROUP BY s.station_id, s.city
    ORDER BY avg_pm10 DESC
    LIMIT 3
""")

q3 = timed_query("Measurement count per pollutant", """
    SELECT p.pollutant_name, COUNT(*) AS n
    FROM measurements m
    JOIN pollutants p ON m.pollutant_id = p.pollutant_id
    GROUP BY p.pollutant_name
    ORDER BY n DESC
""")

q4 = timed_query("Daily peak NO2 for one station", """
    SELECT year, month, day, MAX(value) AS peak_no2
    FROM measurements m
    JOIN pollutants p ON m.pollutant_id = p.pollutant_id
    WHERE p.pollutant_name = 'no2'
      AND m.station_id = (SELECT station_id FROM stations LIMIT 1)
    GROUP BY year, month, day
    ORDER BY year, month, day
""")

⏱  Avg PM2.5 per month: 37.8 ms  (24 rows)
⏱  Top 3 stations by avg PM10: 82.5 ms  (3 rows)
⏱  Measurement count per pollutant: 126.9 ms  (13 rows)
⏱  Daily peak NO2 for one station: 56.7 ms  (716 rows)


In [9]:
q1.head(12)

,year,month,avg_pm25
0,2024,1,181.484169
1,2024,2,95.730308
2,2024,3,77.799548
3,2024,4,75.123728
4,2024,5,97.037308
5,2024,6,67.288539
6,2024,7,38.899614
7,2024,8,28.131560
8,2024,9,44.019754
9,2024,10,120.498857


In [10]:
import os
size_mb = os.path.getsize("air_quality.duckdb") / (1024 * 1024)
print(f"Database file size: {size_mb:.1f} MB")

Database file size: 80.5 MB


In [11]:
start = time.time()
result = con.sql("""
    SELECT year, month, AVG(value) AS avg_pm25
    FROM 'team_4.parquet'
    WHERE pollutant = 'pm25'
    GROUP BY year, month
    ORDER BY year, month
""").df()
elapsed = (time.time() - start) * 1000
print(f"⏱  Same query against parquet directly: {elapsed:.1f} ms")
result.head()

⏱  Same query against parquet directly: 63.4 ms


,year,month,avg_pm25
0,2024,1,181.484169
1,2024,2,95.730308
2,2024,3,77.799548
3,2024,4,75.123728
4,2024,5,97.037308


In [4]:
from google.colab import files
files.download("air_quality.duckdb")

FileNotFoundError: Cannot find file: air_quality.duckdb